In [6]:
import os
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


# Create output folder
os.makedirs("../outputs", exist_ok=True)
#  FETCH ADULT DATASET
print("Fetching Adult dataset...")
adult = fetch_openml(
    "adult",
    version=2,
    as_frame=True
)
df = adult.frame.copy()
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

#  CLEAN DATA
# Replace '?' with NaN
df = df.replace("?", np.nan)

print("\nMissing values converted to NaN.")
#  CREATE TARGET
df["target"] = df["class"].map({
    "<=50K": 0,
    ">50K": 1
}).astype(int)
print("\nTarget distribution:")
print(df["target"].value_counts())

#  FEATURES AND TARGET
X = df.drop(
    columns=["class", "target"]
)
y = df["target"]

# DEFINE FEATURES
numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]
categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

#  NUMERIC PIPELINE
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)
# CATEGORICAL PIPELINE
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)
#  COLUMN TRANSFORMER
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)
#  TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("\n" + "=" * 60)
print("DATA SPLIT")
print("=" * 60)
print("Training data:", X_train.shape)
print("Hold-out test data:", X_test.shape)

#  LOGISTIC REGRESSION PIPELINE
logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

# DECISION TREE PIPELINE

decision_tree_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)
#  TRAIN LOGISTIC MODEL
print("\nTraining Logistic Regression...")
logistic_pipeline.fit(
    X_train,
    y_train
)
print("Logistic Regression training completed!")

#  TRAIN DECISION TREE

print("\nTraining Decision Tree...")

decision_tree_pipeline.fit(
    X_train,
    y_train
)

print("Decision Tree training completed!")

# LOGISTIC REGRESSION INTERPRETABILITY
print("\n" + "=" * 60)
print("LOGISTIC REGRESSION INTERPRETABILITY")
print("=" * 60)


# Get fitted preprocessor
logistic_preprocessor = (
    logistic_pipeline
    .named_steps["preprocessor"]
)

# Get fitted classifier
logistic_classifier = (
    logistic_pipeline
    .named_steps["classifier"]
)

feature_names = (
    logistic_preprocessor
    .get_feature_names_out()
)

coefficients = logistic_classifier.coef_[0]


coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df = (
    coef_df
    .sort_values(
        by="Coefficient",
        ascending=True
    )
    .reset_index(drop=True)
)


print("\nTotal processed features:", len(coef_df))
# Positive coefficients
top_positive = (
    coef_df
    .sort_values(
        by="Coefficient",
        ascending=False
    )
    .head(10)
)

print("\n" + "=" * 60)
print("TOP 10 POSITIVE COEFFICIENTS")
print("=" * 60)

print(
    top_positive.to_string(index=False)
)
top_positive.to_csv(
    "../outputs/top_10_positive_coefficients.csv",
    index=False
)

#Negative Coefficients

top_negative = (
    coef_df
    .sort_values(
        by="Coefficient",
        ascending=True
    )
    .head(10)
)

print("\n" + "=" * 60)
print("TOP 10 NEGATIVE COEFFICIENTS")
print("=" * 60)

print(
    top_negative.to_string(index=False)
)


# Save
top_negative.to_csv(
    "../outputs/top_10_negative_coefficients.csv",
    index=False
)

# INTERPRETATION

print("\n" + "=" * 60)
print("LOGISTIC REGRESSION INTERPRETATION")
print("=" * 60)

print(
    "\nPositive coefficient:"
)

print(
    "A positive coefficient increases the tendency of the "
    "model to predict the >50K income class."
)

print(
    "\nNegative coefficient:"
)

print(
    "A negative coefficient increases the tendency of the "
    "model to predict the <=50K income class."
)


print("\nTop positive features:")

for _, row in top_positive.iterrows():

    print(
        f"- {row['Feature']}: "
        f"{row['Coefficient']:.4f} "
        f"-> associated with >50K income"
    )


print("\nTop negative features:")

for _, row in top_negative.iterrows():

    print(
        f"- {row['Feature']}: "
        f"{row['Coefficient']:.4f} "
        f"-> associated with <=50K income"
    )

coef_df.to_csv(
    "../outputs/all_logistic_coefficients.csv",
    index=False
)

# DECISION TREE INTERPRETABILITY
print("\n" + "=" * 60)
print("DECISION TREE INTERPRETABILITY")
print("=" * 60)
tree_classifier = (
    decision_tree_pipeline
    .named_steps["classifier"]
)

tree_depth = (
    tree_classifier
    .get_depth()
)

tree_leaves = (
    tree_classifier
    .get_n_leaves()
)
tree_train_score = (
    decision_tree_pipeline
    .score(
        X_train,
        y_train
    )
)

tree_test_score = (
    decision_tree_pipeline
    .score(
        X_test,
        y_test
    )
)

accuracy_gap = (
    tree_train_score -
    tree_test_score
)


print("\nTree Depth:", tree_depth)

print(
    "Number of Leaves:",
    tree_leaves
)

print(
    "Training Accuracy:",
    round(tree_train_score, 4)
)

print(
    "Hold-out Test Accuracy:",
    round(tree_test_score, 4)
)

print(
    "Train-Test Accuracy Gap:",
    round(accuracy_gap, 4)
)

# OVERFITTING CHECK
print("\n" + "=" * 60)
print("OVERFITTING CHECK")
print("=" * 60)


if accuracy_gap > 0.10:

    print(
        "Strong evidence of overfitting."
    )

    print(
        "The training accuracy is much higher "
        "than the hold-out test accuracy."
    )

elif accuracy_gap > 0.05:

    print(
        "Some overfitting may be present."
    )

else:

    print(
        "No strong evidence of severe overfitting "
        "based on the accuracy gap."
    )

# DECISION TREE SPLITS


tree_feature_names = (
    decision_tree_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


tree_structure = (
    tree_classifier.tree_
)

split_information = []
# Extract split nodes
for node_id in range(
    tree_structure.node_count
):
    feature_index = (
        tree_structure.feature[node_id]
    )
    if feature_index != -2:

        split_information.append({

            "Node":
                node_id,

            "Feature":
                tree_feature_names[
                    feature_index
                ],

            "Threshold":
                tree_structure.threshold[
                    node_id
                ],

            "Impurity":
                tree_structure.impurity[
                    node_id
                ],

            "Samples":
                tree_structure.n_node_samples[
                    node_id
                ]
        })
split_df = pd.DataFrame(
    split_information
)
top_3_splits = (
    split_df
    .sort_values(
        by="Samples",
        ascending=False
    )
    .head(3)
    .reset_index(drop=True)
)


print("\n" + "=" * 60)
print("TOP 3 DECISION TREE SPLITS")
print("=" * 60)

print(
    top_3_splits.to_string(
        index=False
    )
)

top_3_splits.to_csv(
    "../outputs/top_3_tree_splits.csv",
    index=False
)

print("\n" + "=" * 60)
print("TREE SPLIT INTERPRETATION")
print("=" * 60)


for i, row in top_3_splits.iterrows():

    print(
        f"\nSplit {i + 1}"
    )

    print(
        "Feature:",
        row["Feature"]
    )

    print(
        "Threshold:",
        round(
            row["Threshold"],
            4
        )
    )

    print(
        "Samples:",
        int(row["Samples"])
    )

    print(
        "Interpretation: The tree uses this feature "
        "and threshold to divide observations into "
        "groups with different income-class tendencies."
    )

print("\n" + "=" * 60)
print("TASK 4 FINAL SUMMARY")
print("=" * 60)

print(
    "\nLogistic Regression coefficients provide "
    "feature-level interpretability."
)

print(
    "Positive coefficients are associated with "
    "the >50K class, while negative coefficients "
    "are associated with the <=50K class."
)

print(
    f"\nDecision Tree Depth: {tree_depth}"
)

print(
    f"Decision Tree Training Accuracy: "
    f"{tree_train_score:.4f}"
)

print(
    f"Decision Tree Hold-out Accuracy: "
    f"{tree_test_score:.4f}"
)

print(
    f"Train-Test Gap: {accuracy_gap:.4f}"
)

if accuracy_gap > 0.10:

    print(
        "\nConclusion: The Decision Tree is likely "
        "overfitting the training data."
    )

elif accuracy_gap > 0.05:

    print(
        "\nConclusion: The Decision Tree shows "
        "some signs of overfitting."
    )

else:

    print(
        "\nConclusion: The Decision Tree does not show "
        "strong overfitting based on the accuracy gap."
    )


print(
    "\nTask 4 completed successfully!"
)

print("\nOutput files saved in ../outputs/:")
print("all_logistic_coefficients.csv")
print("top_10_positive_coefficients.csv")
print("top_10_negative_coefficients.csv")
print("top_3_tree_splits.csv")

Fetching Adult dataset...
Dataset loaded successfully!
Dataset shape: (48842, 15)

Missing values converted to NaN.

Target distribution:
target
0    37155
1    11687
Name: count, dtype: int64

DATA SPLIT
Training data: (39073, 14)
Hold-out test data: (9769, 14)

Training Logistic Regression...
Logistic Regression training completed!

Training Decision Tree...
Decision Tree training completed!

LOGISTIC REGRESSION INTERPRETABILITY

Total processed features: 105

TOP 10 POSITIVE COEFFICIENTS
                                       Feature  Coefficient
                         numeric__capital-gain     2.262669
categorical__marital-status_Married-civ-spouse     1.556920
 categorical__marital-status_Married-AF-spouse     1.260689
           categorical__native-country_Ireland     0.923063
            categorical__native-country_France     0.921465
          categorical__native-country_Cambodia     0.839293
       categorical__occupation_Exec-managerial     0.786458
           categorical__